# Exploratory Data Analysis — 따릉이 수요 예측
모델링 결정사항을 데이터로 정당화합니다.
- S=24 계절 주기
- 평일 / 주말 분리
- 2023+ 데이터 사용
- 기온 · 강수 외생변수

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..').resolve()))

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

from src.config import DATA_DIR, TARGET_RENT_IDS, TARGET_RENT_ID
from src.utils import load_filtered_csvs, make_series, load_weather

PALETTE = sns.color_palette('tab10')

## 1. 데이터 개요

In [ ]:

df_raw = load_filtered_csvs(DATA_DIR)          # 전체 기간 로드
df_2023 = df_raw[df_raw['datetime'].dt.year >= 2023]

try:
    from src.utils import load_weather_full
    weather_df = load_weather_full(DATA_DIR)
    HAS_RAIN = 'rainfall' in weather_df.columns
except Exception:
    weather_df = load_weather(DATA_DIR)
    HAS_RAIN = False

print('=== 대여 데이터 ===')
print(f'기간   : {df_raw["datetime"].min().date()} ~ {df_raw["datetime"].max().date()}')
print(f'정류소 : {df_raw["RENT_ID"].nunique()}개')
print(f'총 레코드: {len(df_raw):,}건')
print(f'\n=== 기상 데이터 ===')
print(f'기간   : {weather_df.index.min().date()} ~ {weather_df.index.max().date()}')
print(f'컬럼   : {weather_df.columns.tolist()}')
df_raw.head()


## 2. 결측치 분석
기존 `missing_value_report.md` 핵심 내용 요약

In [1]:
# 정류소별 대여 기록이 없는 날 (하루 전체 결측) 계산
df_raw['date'] = df_raw['datetime'].dt.date
daily_counts = df_raw.groupby(['RENT_ID', 'date']).size().reset_index(name='records')

all_dates = pd.date_range(df_raw['datetime'].min().date(),
                           df_raw['datetime'].max().date(), freq='D')
total_days = len(all_dates)

missing_summary = []
for rid in TARGET_RENT_IDS:
    observed = daily_counts[daily_counts['RENT_ID'] == rid]['date'].nunique()
    missing_days = total_days - observed
    missing_summary.append({'station': rid, 'missing_days': missing_days,
                             'missing_pct': missing_days / total_days * 100})

miss_df = pd.DataFrame(missing_summary).sort_values('missing_pct', ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(miss_df['station'], miss_df['missing_pct'],
              color=['tomato' if p > 5 else 'steelblue' for p in miss_df['missing_pct']])
ax.axhline(2.1, color='gray', linestyle='--', linewidth=0.8, label='전체 평균 2.1%')
ax.set_xlabel('Station ID')
ax.set_ylabel('결측률 (%)')
ax.set_title('정류소별 일별 결측률 (하루 전체 결측 기준)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('결측률 상위 정류소:')
print(miss_df.head(8).to_string(index=False))

NameError: name 'df_raw' is not defined

## 3. 전체 시계열 추이 — 2023+ 데이터 사용 정당화

In [ ]:
# 전체 23개 정류소 월별 총 대여량
df_raw['yearmonth'] = df_raw['datetime'].dt.to_period('M')
monthly = df_raw.groupby('yearmonth')['CNT'].sum().reset_index()
monthly['yearmonth_dt'] = monthly['yearmonth'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly['yearmonth_dt'], monthly['CNT'], color='steelblue', linewidth=1.5)
ax.fill_between(monthly['yearmonth_dt'], monthly['CNT'], alpha=0.15, color='steelblue')

# COVID 구간 음영
ax.axvspan(pd.Timestamp('2021-01-01'), pd.Timestamp('2022-12-31'),
           alpha=0.12, color='red', label='COVID 기간 (2021~2022)')
ax.axvline(pd.Timestamp('2023-01-01'), color='red', linestyle='--',
           linewidth=1.5, label='학습 시작점 (2023-01)')

ax.set_title('23개 정류소 월별 총 대여량 (2021~2025)')
ax.set_ylabel('월별 대여량')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=30)
ax.legend()
plt.tight_layout()
plt.show()

## 4. 일별·주별 패턴 — S=24 정당화 + 평일/주말 분리 정당화

In [ ]:
# 2023+ 대표 정류소(02128) 시계열
series = make_series(df_raw[df_raw['datetime'].dt.year >= 2023], TARGET_RENT_ID)

df_pat = pd.DataFrame({'CNT': series})
df_pat['hour'] = df_pat.index.hour
df_pat['dayofweek'] = df_pat.index.dayofweek
df_pat['is_weekend'] = df_pat['dayofweek'] >= 5
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 평일 vs 주말 시간대별 평균
for is_wknd, label, color in [(False, 'Weekday', 'steelblue'), (True, 'Weekend', 'tomato')]:
    hourly = df_pat[df_pat['is_weekend'] == is_wknd].groupby('hour')['CNT'].mean()
    axes[0].plot(hourly.index, hourly.values, label=label, color=color, linewidth=2, marker='o', markersize=3)

axes[0].set_title('시간대별 평균 수요 — 평일 vs 주말\n(S=24 정당화)')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Average CNT')
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend()

# 요일별 일평균
daily_dow = df_pat.groupby(['dayofweek', 'hour'])['CNT'].mean().groupby('dayofweek').sum()
colors = ['steelblue']*5 + ['tomato']*2
axes[1].bar([dow_names[i] for i in daily_dow.index], daily_dow.values, color=colors)
axes[1].set_title('요일별 일 총 수요\n(평일/주말 분리 정당화)')
axes[1].set_ylabel('Daily Total CNT')

plt.tight_layout()
plt.show()

## 5. ACF / PACF — S=24 계절 주기 정당화

In [ ]:
series_wd = series[series.index.dayofweek < 5]  # 평일

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
plot_acf(series_wd.dropna(), lags=72, ax=axes[0],
         title='ACF — 평일 시계열 (lag 72시간)')
plot_pacf(series_wd.dropna(), lags=72, ax=axes[1],
          title='PACF — 평일 시계열 (lag 72시간)')

# lag 24, 48 강조
for ax in axes:
    ax.axvline(24, color='red', linestyle='--', alpha=0.6, linewidth=1, label='lag 24')
    ax.axvline(48, color='orange', linestyle='--', alpha=0.6, linewidth=1, label='lag 48')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print('→ lag 24, 48에서 유의한 스파이크 → S=24 (일간 계절성) 확인')

## 6. 기상 변수 — 외생변수 정당화

In [ ]:

series = make_series(df_2023, TARGET_RENT_ID)

df_w = pd.DataFrame({'CNT': series})
df_w['temp'] = weather_df['temp'].reindex(df_w.index).ffill()

if HAS_RAIN:
    rain_col = 'rainfall' if 'rainfall' in weather_df.columns else 'rain'
    df_w['rain'] = weather_df[rain_col].reindex(df_w.index).fillna(0)
    df_w['is_rain'] = df_w['rain'] > 1.0
else:
    df_w['is_rain'] = False

df_w = df_w.dropna()

fig, axes = plt.subplots(1, 2 + HAS_RAIN, figsize=(14, 4))

# 기온 vs 수요
sample = df_w.sample(min(3000, len(df_w)), random_state=42)
axes[0].scatter(sample['temp'], sample['CNT'], alpha=0.1, s=5, color='steelblue')
z = np.polyfit(sample['temp'], sample['CNT'], 2)
t_range = np.linspace(sample['temp'].min(), sample['temp'].max(), 100)
axes[0].plot(t_range, np.poly1d(z)(t_range), color='red', linewidth=2)
axes[0].set_xlabel('기온 (°C)')
axes[0].set_ylabel('대여량')
axes[0].set_title('기온 vs 대여량')

# 월별 평균
df_w['month'] = df_w.index.month
monthly_avg = df_w.groupby('month')[['CNT','temp']].mean()
ax2b = axes[1].twinx()
axes[1].bar(monthly_avg.index, monthly_avg['CNT'], color='steelblue', alpha=0.6)
ax2b.plot(monthly_avg.index, monthly_avg['temp'], color='red', marker='o', linewidth=2)
axes[1].set_xlabel('월')
axes[1].set_ylabel('평균 대여량', color='steelblue')
ax2b.set_ylabel('평균 기온 (°C)', color='red')
axes[1].set_title('월별 수요 & 기온')
axes[1].set_xticks(range(1,13))

# 강수 여부별 수요
if HAS_RAIN:
    rain_avg = df_w.groupby('is_rain')['CNT'].mean()
    bars = axes[2].bar(['비 없음', '비 있음 (>1mm)'], rain_avg.values,
                        color=['steelblue', 'gray'])
    for bar, val in zip(bars, rain_avg.values):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                     f'{val:.2f}', ha='center', va='bottom', fontsize=10)
    axes[2].set_ylabel('평균 대여량')
    axes[2].set_title('강수 여부별 평균 수요')

plt.tight_layout()
plt.show()


## 7. 23개 정류소 수요 분포 비교

In [ ]:
# 2023+ 정류소별 시간당 평균 대여량
df_2023 = df_raw[df_raw['datetime'].dt.year >= 2023]
station_avg = (
    df_2023.groupby('RENT_ID')['CNT'].agg(['mean','sum','std'])
    .reindex(TARGET_RENT_IDS)
    .sort_values('mean', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].barh(station_avg.index, station_avg['mean'], color='steelblue')
axes[0].set_xlabel('시간당 평균 대여량')
axes[0].set_title('정류소별 평균 수요 (2023+)')
axes[0].invert_yaxis()

# 박스플롯 (상위 10개)
top10 = station_avg.head(10).index.tolist()
box_data = [make_series(df_2023[df_2023['datetime'].dt.year >= 2023], rid).dropna().values
            for rid in top10]
axes[1].boxplot(box_data, labels=top10, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.5))
axes[1].set_xlabel('Station ID')
axes[1].set_ylabel('시간당 대여량')
axes[1].set_title('수요 상위 10개 정류소 분포')
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

print(station_avg.round(3).to_string())

## 8. 대표 정류소 (02128) 시계열 분해

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# 일별 집계 후 분해 (시간별로 하면 너무 노이즈)
series_daily = series.resample('D').sum()
series_daily = series_daily['2023':]

decomp = seasonal_decompose(series_daily.dropna(), model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title='Observed (일별 대여량)')
decomp.trend.plot(ax=axes[1], title='Trend')
decomp.seasonal.plot(ax=axes[2], title='Seasonal (주간 패턴)')
decomp.resid.plot(ax=axes[3], title='Residual')
for ax in axes:
    ax.set_ylabel('')
plt.suptitle(f'시계열 분해 — Station {TARGET_RENT_ID} (2023+, 주기=7일)', y=1.01)
plt.tight_layout()
plt.show()

## 요약 — 모델링 결정 근거

| 결정 | 근거 |
|------|------|
| **S=24 고정** | ACF/PACF lag 24, 48에서 유의한 피크 |
| **평일/주말 분리** | 출퇴근(평일) vs 여가(주말) 시간대 패턴이 구조적으로 다름 |
| **2023+ 사용** | 2021~2022 COVID 기간 수요 패턴이 비정상적으로 낮음 |
| **기온 외생변수** | 기온-수요 2차 관계 (추울수록↓, 너무 더워도↓, 봄가을↑) |
| **강수 외생변수** | 비 오는 날 평균 수요 유의미하게 감소 |